In [18]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer

# -----------------------
# CONFIG
# -----------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_PATH = "roberta_base_finetuned_dualhead.pt"

# -----------------------
# LOAD CHECKPOINT
# -----------------------
checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)

MODEL_NAME = checkpoint["model_name"]

print(f"Loaded checkpoint from: {MODEL_PATH}")
print(f"Base model: {MODEL_NAME}")

# -----------------------
# MODEL DEFINITION
# -----------------------
class MultiTaskModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(MODEL_NAME)
        hidden = self.encoder.config.hidden_size

        # Enable gradient checkpointing
        if hasattr(self.encoder, "gradient_checkpointing_enable"):
            self.encoder.gradient_checkpointing_enable()

        # Feature layer
        self.feature_layer = nn.Sequential(
            nn.Linear(hidden * 2, hidden),
            nn.LayerNorm(hidden),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # Binary head
        self.binary_head = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden // 2, 1)
        )

        # Severity head
        self.severity_head = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden // 2, 4)
        )

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)

        cls = out.last_hidden_state[:, 0]

        mean_pool = (out.last_hidden_state * attention_mask.unsqueeze(-1)).sum(1)
        mean_pool = mean_pool / attention_mask.sum(1, keepdim=True)

        combined = torch.cat([cls, mean_pool], dim=1)

        features = self.feature_layer(combined)

        binary = self.binary_head(features).squeeze(-1)
        severity = self.severity_head(features)

        return binary, severity


# -----------------------
# LOAD MODEL
# -----------------------
model = MultiTaskModel().to(DEVICE)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Model loaded successfully")

# -----------------------
# LOAD TOKENIZER
# -----------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# -----------------------
# PREDICTION FUNCTION
# -----------------------
def predict(text):
    model.eval()

    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    ).to(DEVICE)

    with torch.no_grad():
        binary_logits, severity_logits = model(
            enc["input_ids"], enc["attention_mask"]
        )

        prob = torch.sigmoid(binary_logits).item()
        severity = torch.argmax(severity_logits, dim=1).item()

    return {
        "text": text,
        "is_hate": bool(prob > 0.5),
        "confidence": round(prob, 4),
        "severity": severity
    }


# -----------------------
# TEST
# -----------------------
tests = [
    "Have a nice day",
    "You are stupid",
    "I hate you",
    "I will kill you"
]

for t in tests:
    print(predict(t))

Loaded checkpoint from: roberta_base_finetuned_dualhead.pt
Base model: roberta-base


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully
{'text': 'Have a nice day', 'is_hate': False, 'confidence': 0.0004, 'severity': 0}
{'text': 'You are stupid', 'is_hate': True, 'confidence': 0.9995, 'severity': 2}
{'text': 'I hate you', 'is_hate': True, 'confidence': 0.9988, 'severity': 2}
{'text': 'I will kill you', 'is_hate': True, 'confidence': 0.9999, 'severity': 3}
